# Домашнє завдання 9. Агент технічної підтримки на LangGraph

Агент послідовно проходить чотири вузли: `classify_query` → `search_faq` → `draft_response` → `check_escalation`.

### Встановлення пакетів

In [ ]:
!pip install -q -U "langgraph>=1.0" langchain langchain-openai langchain-core
import warnings
warnings.filterwarnings("ignore")

### Налаштування API ключа

In [ ]:
import os
from getpass import getpass

def get_secret(name: str) -> str:
    # 1) змінна середовища, 2) Colab Secrets, 3) ручне введення
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return getpass(f"Введіть {name}: ")

os.environ["OPENROUTER_API_KEY"] = get_secret("OPENROUTER_API_KEY")

Введіть OPENROUTER_API_KEY: ··········


### Типи даних

In [ ]:
from typing import TypedDict, Literal, Optional, List

class QueryClassification(TypedDict):
    """Класифікація запиту до служби технічної підтримки"""
    type: Literal["problem", "question", "complaint"]
    category: Literal["password", "payment", "technical", "account", "general"]
    urgency: Literal["low", "medium", "high"]
    summary: str

class SupportAgentState(TypedDict):
    """Стан агента"""
    user_query: str
    classification: Optional[QueryClassification]
    search_results: Optional[List[str]]
    draft_response: Optional[str]
    needs_escalation: bool

### База знань FAQ

In [ ]:
FAQ_DATABASE = {
    "password": [
        "Скидання пароля: Натисніть 'Забули пароль?' → введіть email → перевірте пошту",
        "Новий пароль: мінімум 8 символів з літерами та цифрами",
        "Лист не прийшов - перевірте папку спам"
    ],
    "payment": [
        "Перевірте дані картки та баланс",
        "Подвійне списання повернеться автоматично за 3-5 днів",
        "Зміна тарифу: Профіль → Підписка"
    ],
    "technical": [
        "Оновіть сторінку (F5) або очистьте кеш",
        "Спробуйте інший браузер",
        "Перевірте status.ourservice.com"
    ],
    "account": [
        "Перевірте правильність email",
        "Корпоративні акаунти: розділ 'Управління командою'",
        "Блокування - зверніться до підтримки"
    ],
    "general": [
        "Підтримка працює 24/7",
        "Час відповіді: 2-4 години"
    ]
}

### Ініціалізація моделі

Беру `gpt-4o-mini` через OpenRouter — дешева модель, для цього завдання її достатньо.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="openai/gpt-4o-mini",
    temperature=0
)

### Вузол 1: класифікація запиту

In [ ]:
def classify_query(state: SupportAgentState) -> dict:
    structured_llm = llm.with_structured_output(QueryClassification)

    classification_prompt = f"""
    Ти класифікуєш запити до служби технічної підтримки.

    Запит користувача: {state['user_query']}

    Правила для type:
    - complaint — користувач обурений, скаржиться, звинувачує сервіс або вимагає щось негайно
      (знаки оклику, "шахрайство", "коли полагодите?")
    - problem — у користувача щось не виходить (наприклад, забув пароль, не може увійти)
      і він спокійно просить допомогти
    - question — користувач питає, чи можна щось зробити або як щось зробити, проблеми немає

    Правила для category:
    - password — пароль, вхід в акаунт, відновлення доступу
    - payment — оплата, списання коштів, підписка, тарифні плани, зміна тарифу
    - technical — сайт або застосунок не працює, помилки, збої
    - account — налаштування акаунту, email, користувачі, корпоративні акаунти
    - general — все інше

    Правила для urgency:
    - high — скарга, проблеми з грошима або сервіс взагалі не працює
    - medium — проблема, яка заважає користувачу (наприклад, не може увійти)
    - low — звичайне питання без проблеми

    summary — одне коротке речення українською мовою про суть запиту.
    """

    classification = structured_llm.invoke(classification_prompt)
    return {"classification": classification}

### Вузол 2: пошук у базі FAQ

Беремо пункти з розділу FAQ для категорії і просимо модель обрати 1–2 найкорисніші для цього запиту.
Пошук виконується для кожного запиту, навіть якщо потім буде ескалація.

In [ ]:
class FAQSelection(TypedDict):
    """Номери найрелевантніших пунктів FAQ"""
    indices: List[int]

def search_faq(state: SupportAgentState) -> dict:
    category = state["classification"]["category"]
    faq_items = FAQ_DATABASE.get(category, FAQ_DATABASE["general"])
    numbered_faq = "\n".join(f"{i}. {item}" for i, item in enumerate(faq_items, 1))

    search_prompt = f"""
    Запит користувача: {state['user_query']}

    Пункти бази знань:
    {numbered_faq}

    Обери 1 або 2 пункти, які найкраще допоможуть відповісти саме на цей запит.
    Якщо достатньо одного пункту — поверни один. Номери повертай у порядку релевантності.
    """

    selection = llm.with_structured_output(FAQSelection).invoke(search_prompt)

    # залишаємо тільки існуючі номери, без повторів, максимум 2
    indices = [i for i in selection.get("indices", []) if 1 <= i <= len(faq_items)]
    indices = list(dict.fromkeys(indices))[:2]
    if not indices:
        indices = [1]

    return {"search_results": [faq_items[i - 1] for i in indices]}

### Вузол 3: генерація відповіді

Обмежую відповідь 35 словами в промпті і обрізаю в коді, якщо модель написала більше 40.
Так навіть разом із повідомленням про ескалацію відповідь не перевищить 50 слів.

In [ ]:
MAX_DRAFT_WORDS = 40

def draft_response(state: SupportAgentState) -> dict:
    classification = state["classification"]
    faq_text = "\n".join(f"- {item}" for item in state.get("search_results") or [])

    draft_prompt = f"""
    Ти агент технічної підтримки. Напиши відповідь користувачу українською мовою.

    Запит: {state['user_query']}
    Тип запиту: {classification['type']}
    Суть: {classification['summary']}

    Інформація з бази знань:
    {faq_text}

    Правила:
    - максимум 35 слів
    - одразу конкретні кроки, без привітань і вступних фраз
    - без ввічливих фраз-заповнювачів (будь ласка, дякуємо, якщо проблема залишається, повідомте)
    - використовуй тільки інформацію з бази знань, нічого не вигадуй
    """

    response = llm.invoke(draft_prompt).content.strip()

    words = response.split()
    if len(words) > MAX_DRAFT_WORDS:
        response = " ".join(words[:MAX_DRAFT_WORDS]) + "..."

    return {"draft_response": response}

### Вузол 4: перевірка ескалації

Ескалація, якщо `urgency == "high"`, або `type == "complaint"`, або в запиті є слова «шахрайство», «судов», «юридичн».
Для ескальованих запитів до відповіді додається повідомлення про передачу спеціалісту.

In [ ]:
ESCALATION_KEYWORDS = ["шахрайство", "судов", "юридичн"]
ESCALATION_MESSAGE = "⚠️ Запит передано спеціалісту (відповідь за 24 год)."

def check_escalation(state: SupportAgentState) -> dict:
    classification = state["classification"]
    query = state["user_query"].lower()

    has_keywords = any(word in query for word in ESCALATION_KEYWORDS)
    needs_escalation = (
        classification["urgency"] == "high"
        or classification["type"] == "complaint"
        or has_keywords
    )

    response = state["draft_response"]
    if needs_escalation:
        response = f"{response}\n\n{ESCALATION_MESSAGE}"

    return {"needs_escalation": needs_escalation, "draft_response": response}

### Побудова графа

In [ ]:
def build_support_agent():
    workflow = StateGraph(SupportAgentState)

    workflow.add_node("classify_query", classify_query)
    workflow.add_node("search_faq", search_faq)
    workflow.add_node("draft_response", draft_response)
    workflow.add_node("check_escalation", check_escalation)

    workflow.add_edge(START, "classify_query")
    workflow.add_edge("classify_query", "search_faq")
    workflow.add_edge("search_faq", "draft_response")
    workflow.add_edge("draft_response", "check_escalation")
    workflow.add_edge("check_escalation", END)

    return workflow.compile(checkpointer=MemorySaver())

Перевіряю структуру графа

In [ ]:
print(build_support_agent().get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify_query(classify_query)
	search_faq(search_faq)
	draft_response(draft_response)
	check_escalation(check_escalation)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify_query;
	classify_query --> search_faq;
	draft_response --> check_escalation;
	search_faq --> draft_response;
	check_escalation --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Оцінка релевантності та тестування

In [ ]:
def calculate_answer_relevancy(query: str, response: str) -> float:
    """Оцінка релевантності"""

    relevancy_prompt = f"""
    Оцінка релевантності (0-1):
    Запит: {query}
    Відповідь: {response}

    Тільки число:
    """

    try:
        result = llm.invoke(relevancy_prompt)
        text = result.content.strip()
        score = float(''.join(c for c in text if c.isdigit() or c == '.'))
        return min(max(score, 0.0), 1.0)
    except:
        return 0.5

def test_agent():
    """Тестування агента на 5 запитах"""

    agent = build_support_agent()

    test_queries = [
        "Я забув пароль від свого акаунту, як його відновити?",
        "З мене двічі зняли гроші за підписку! Це шахрайство! Поверніть кошти негайно!",
        "Чи можу я змінити тарифний план на дешевший?",
        "Сайт не працює вже третю годину! Коли полагодите?",
        "Як додати нового користувача до корпоративного акаунту?"
    ]

    results = []
    relevancy_scores = []

    print("=" * 70)
    print("ТЕСТУВАННЯ АГЕНТА ТЕХНІЧНОЇ ПІДТРИМКИ")
    print("=" * 70)

    for i, query in enumerate(test_queries, 1):
        print(f"\n{'='*70}")
        print(f"📧 ТЕСТ {i}/5")
        print(f"{'='*70}")
        print(f"ЗАПИТ: {query}")
        print("-" * 70)

        initial_state = {
            "user_query": query,
            "needs_escalation": False
        }

        config = {"configurable": {"thread_id": f"test_{i}"}}

        try:
            result = agent.invoke(initial_state, config)
            results.append(result)

            # Виводимо класифікацію
            if result.get("classification"):
                cls = result["classification"]
                print(f"\n📊 КЛАСИФІКАЦІЯ:")
                print(f"   Тип: {cls.get('type')}")
                print(f"   Категорія: {cls.get('category')}")
                print(f"   Терміновість: {cls.get('urgency')}")
                print(f"   Резюме: {cls.get('summary')}")

            # Виводимо знайдені FAQ
            if result.get("search_results"):
                print(f"\n🔍 FAQ ({len(result['search_results'])} пунктів):")
                for j, faq in enumerate(result['search_results'], 1):
                    print(f"   {j}. {faq}")

            # ВИВОДИМО ВІДПОВІДЬ АГЕНТА
            if result.get("draft_response"):
                print(f"\n💬 ВІДПОВІДЬ АГЕНТА:")
                print("-" * 70)
                print(result["draft_response"])
                print("-" * 70)

                # Рахуємо релевантність
                relevancy = calculate_answer_relevancy(query, result["draft_response"])
                relevancy_scores.append(relevancy)
                print(f"\n📈 Релевантність: {relevancy:.0%}")

            # Статус ескалації
            if result.get("needs_escalation"):
                print("\n⚠️ СТАТУС: Ескаловано до спеціаліста")
            else:
                print("\n✅ СТАТУС: Оброблено автоматично")

        except Exception as e:
            print(f"❌ ПОМИЛКА: {e}")
            results.append({})

    # Фінальні метрики
    print(f"\n{'='*70}")
    print("ПІДСУМКОВІ МЕТРИКИ")
    print("=" * 70)

    # Task Completion Rate
    completed = sum(1 for r in results if r.get("draft_response"))
    tcr = completed / len(results) if results else 0
    print(f"✅ Task Completion Rate: {tcr:.0%} ({completed}/{len(results)})")

    # Answer Relevancy
    if relevancy_scores:
        avg_relevancy = sum(relevancy_scores) / len(relevancy_scores)
        print(f"🎯 Average Answer Relevancy: {avg_relevancy:.0%}")

    # Escalation Rate
    escalated = sum(1 for r in results if r.get("needs_escalation"))
    print(f"📊 Escalation Rate: {escalated}/{len(results)} ({escalated/len(results)*100:.0f}%)")

    print("=" * 70)

    return results

In [ ]:
results = test_agent()

ТЕСТУВАННЯ АГЕНТА ТЕХНІЧНОЇ ПІДТРИМКИ

📧 ТЕСТ 1/5
ЗАПИТ: Я забув пароль від свого акаунту, як його відновити?
----------------------------------------------------------------------

📊 КЛАСИФІКАЦІЯ:
   Тип: problem
   Категорія: password
   Терміновість: medium
   Резюме: Користувач забув пароль і просить допомогу в його відновленні.

🔍 FAQ (2 пунктів):
   1. Скидання пароля: Натисніть 'Забули пароль?' → введіть email → перевірте пошту
   2. Лист не прийшов - перевірте папку спам

💬 ВІДПОВІДЬ АГЕНТА:
----------------------------------------------------------------------
Натисніть 'Забули пароль?', введіть ваш email та перевірте пошту. Якщо лист не прийшов, перевірте папку спам.
----------------------------------------------------------------------

📈 Релевантність: 100%

✅ СТАТУС: Оброблено автоматично

📧 ТЕСТ 2/5
ЗАПИТ: З мене двічі зняли гроші за підписку! Це шахрайство! Поверніть кошти негайно!
----------------------------------------------------------------------

📊 КЛАСИФІКАЦІЯ:
  

Додатково перевіряю кількість слів у кожній відповіді

In [ ]:
for i, r in enumerate(results, 1):
    words = len(r.get("draft_response", "").split())
    print(f"Запит {i}: {words} слів, ескалація = {r.get('needs_escalation')}")

Запит 1: 16 слів, ескалація = False
Запит 2: 23 слів, ескалація = True
Запит 3: 20 слів, ескалація = False
Запит 4: 27 слів, ескалація = True
Запит 5: 15 слів, ескалація = False
